# Smoke test del training

Questo notebook esegue un addestramento reale su un campione ridotto del dataset. Gli artefatti vengono scritti nella cartella smoke_artifacts, senza modificare i modelli e i file persistenti usati dall'applicazione.

## Preparazione dell'ambiente

In locale il notebook usa il repository corrente. In Google Colab clona il progetto e installa le dipendenze.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/RPBrando/company_sentiment.git"

if not Path("src").exists():
    repository_path = Path("company_sentiment")
    if not repository_path.exists():
        subprocess.run(["git", "clone", REPOSITORY_URL], check=True)
    os.chdir(repository_path)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
        check=True,
    )

print(f"Repository: {Path.cwd()}")

Repository: /Users/hanus/Documents/GitHub/company_sentiment


In [2]:
from collections import Counter
import shutil

SMOKE_ROOT = Path("smoke_artifacts").resolve()
if SMOKE_ROOT.exists():
    shutil.rmtree(SMOKE_ROOT)

os.environ["PERSISTENT_DATA_PATH"] = str(SMOKE_ROOT / "data")
os.environ["HF_MODEL_PATH"] = str(SMOKE_ROOT / "models")
os.environ["HF_DATASET_PATH"] = str(
    (Path.cwd() / "data" / "sentiment_datasets").resolve()
)

from src import csvfiles_manager, model, newdata_manager

SAMPLE_SIZE = 300
MODEL_VERSION = "smoke_test_v1"

dataset = newdata_manager.Dataset("train.csv")
X_train, X_test, y_train, y_test = dataset.return_training_data(
    sample_size=SAMPLE_SIZE
)

print(f"Righe di training: {len(X_train)}")
print(f"Righe di test: {len(X_test)}")
print(f"Distribuzione classi: {dict(Counter(y_train + y_test))}")

/Users/hanus/Documents/GitHub/company_sentiment/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Righe di training: 240
Righe di test: 60
Distribuzione classi: {0: 83, 2: 88, 1: 129}


## Addestramento ridotto

Il test usa il modello base previsto dalla traccia e una sola epoca, come configurato nel progetto. Lo scopo non e ottenere il modello migliore, ma verificare che caricamento, training, valutazione e salvataggio funzionino insieme.

In [3]:
trained_model, evaluations = model.train_model(
    model_trained=model.BASEMODEL_NAME,
    dataset_name="train.csv",
    new_version_name=MODEL_VERSION,
    sample_size=SAMPLE_SIZE,
)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 58113.68it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/Users/hanus/Documents/GitHub/company_sentiment/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Training Accuracy,Training Precision,Training Recall,Training F1
0.549872,No log,0,0.683333,0.695198,0.736048,0.680098


/Users/hanus/Documents/GitHub/company_sentiment/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Training Loss,Validation Loss,Step,Training Accuracy,Training Precision,Training Recall,Training F1
0.429200,No log,30,0.866667,0.882576,0.877074,0.871674


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.83it/s]


modello salvato correttamente
file scritto correttamente


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 24114.85it/s]

Il modello migliore è: smoke_test_v1
Il modello migliore è: smoke_test_v1


In [4]:
model_directory = Path(os.environ["HF_MODEL_PATH"]) / MODEL_VERSION

assert trained_model is not None
assert (model_directory / "config.json").exists()
assert (model_directory / "tokenizer_config.json").exists()
assert evaluations["before training"]
assert evaluations["after training"]
assert csvfiles_manager.MODELTRAININGFILE_COMPARE.exists()

print("Smoke test completato correttamente.")
print(f"Modello salvato in: {model_directory}")
print(
    f"F1 prima: {evaluations['before training'].get('before_training_f1', 'non disponibile')}"
)
print(
    f"F1 dopo:  {evaluations['after training'].get('after_training_f1', 'non disponibile')}"
)

Smoke test completato correttamente.
Modello salvato in: /Users/hanus/Documents/GitHub/company_sentiment/smoke_artifacts/models/smoke_test_v1
F1 prima: 0.68009768009768
F1 dopo:  0.8716735594154948


## Risultato atteso

Se l'ultima cella termina senza errori, il flusso minimo di addestramento e valido: il dataset viene letto, il modello viene valutato e addestrato, la nuova versione viene salvata e le metriche vengono registrate.